In [1]:
import pandas as pd

In [4]:
df=pd.read_csv('Cleaned_data.csv', header=None)

C:\Users\Aashli\AppData\Local\Temp\ipykernel_28844\3776656027.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('Cleaned_data.csv', header=None)


In [5]:
print(df.shape)

(1033036, 8)


In [11]:
print(df.head())

        0      1                                   2   3  \
0  489434  21232      STRAWBERRY CERAMIC TRINKET BOX  24   
1  489434  21523  FANCY FONT HOME SWEET HOME DOORMAT  10   
2  489434  21871                 SAVE THE PLANET MUG  24   
3  489434  22041        RECORD FRAME 7" SINGLE SIZE   48   
4  489434  22064          PINK DOUGHNUT TRINKET POT   24   

                             4     5        6               7  
0  2009-12-01 07:45:00.0000000  1.25  13085.0  United Kingdom  
1  2009-12-01 07:45:00.0000000  5.95  13085.0  United Kingdom  
2  2009-12-01 07:45:00.0000000  1.25  13085.0  United Kingdom  
3  2009-12-01 07:45:00.0000000  2.10  13085.0  United Kingdom  
4  2009-12-01 07:45:00.0000000  1.65  13085.0  United Kingdom  


In [12]:
print(df.shape)
print("Number of columns:", len(df.columns))
print(df.head())

(1033036, 8)
Number of columns: 8
        0      1                                   2   3  \
0  489434  21232      STRAWBERRY CERAMIC TRINKET BOX  24   
1  489434  21523  FANCY FONT HOME SWEET HOME DOORMAT  10   
2  489434  21871                 SAVE THE PLANET MUG  24   
3  489434  22041        RECORD FRAME 7" SINGLE SIZE   48   
4  489434  22064          PINK DOUGHNUT TRINKET POT   24   

                             4     5        6               7  
0  2009-12-01 07:45:00.0000000  1.25  13085.0  United Kingdom  
1  2009-12-01 07:45:00.0000000  5.95  13085.0  United Kingdom  
2  2009-12-01 07:45:00.0000000  1.25  13085.0  United Kingdom  
3  2009-12-01 07:45:00.0000000  2.10  13085.0  United Kingdom  
4  2009-12-01 07:45:00.0000000  1.65  13085.0  United Kingdom  


In [13]:
df.columns = [
    'Invoice',
    'StockCode',
    'Description',
    'Quantity',
    'InvoiceDate',
    'Price',
    'Customer_ID',
    'Country'
]

In [19]:
# Convert all object columns to string except Invoice

cols = df.select_dtypes(include=['object']).columns.drop('Invoice')

df[cols] = df[cols].astype('string')

# Check updated data types
df.dtypes        
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1033036 non-null  object 
 1   StockCode    1033036 non-null  string 
 2   Description  1028761 non-null  string 
 3   Quantity     1033036 non-null  int64  
 4   InvoiceDate  1033036 non-null  string 
 5   Price        1033036 non-null  float64
 6   Customer_ID  797885 non-null   float64
 7   Country      1033036 non-null  string 
dtypes: float64(2), int64(1), object(1), string(4)
memory usage: 63.1+ MB


In [20]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  string        
 2   Description  1028761 non-null  string        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  string        
dtypes: datetime64[ns](1), float64(2), int64(1), object(1), string(3)
memory usage: 63.1+ MB


In [22]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY

rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()

snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

In [23]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY

rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()

snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

In [24]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

df['Revenue'] = df['Quantity'] * df['Price']


df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary',
         'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID', how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)

KeyError: "['R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned'] not in index"

In [25]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

rfm['R_Score'] = pd.qcut(
    rfm['Recency'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


def segment_customer(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'

    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm['IsChurned'] = rfm['Recency'] > 90


In [26]:
print(rfm.head())

print("\nRFM Columns:")
print(rfm.columns.tolist())

print("\nCustomer Segments:")
print(rfm['Segment'].value_counts())

print("\nChurn Distribution:")
print(rfm['IsChurned'].value_counts())

   Customer_ID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0      12346.0      325         12  77556.46        2        5        5   
1      12347.0        2          8   4921.53        5        4        5   
2      12348.0       75          5   2019.40        3        4        4   
3      12349.0       18          4   4428.69        5        3        5   
4      12350.0      310          1    334.40        2        1        2   

     Segment  IsChurned  
0    At Risk       True  
1  Champions      False  
2      Loyal      False  
3      Loyal      False  
4       Lost       True  

RFM Columns:
['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']

Customer Segments:
Segment
Lost               1529
Loyal              1416
Champions          1289
At Risk             823
New                 442
Needs Attention     382
Name: count, dtype: int64

Churn Distribution:
IsChurned
True     2991
False    2890
Name: count, 

In [27]:
print(rfm.head())

print("\nRFM Columns:")
print(rfm.columns.tolist())

print("\nCustomer Segments:")
print(rfm['Segment'].value_counts())

print("\nChurn Distribution:")
print(rfm['IsChurned'].value_counts())

   Customer_ID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0      12346.0      325         12  77556.46        2        5        5   
1      12347.0        2          8   4921.53        5        4        5   
2      12348.0       75          5   2019.40        3        4        4   
3      12349.0       18          4   4428.69        5        3        5   
4      12350.0      310          1    334.40        2        1        2   

     Segment  IsChurned  
0    At Risk       True  
1  Champions      False  
2      Loyal      False  
3      Loyal      False  
4       Lost       True  

RFM Columns:
['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']

Customer Segments:
Segment
Lost               1529
Loyal              1416
Champions          1289
At Risk             823
New                 442
Needs Attention     382
Name: count, dtype: int64

Churn Distribution:
IsChurned
True     2991
False    2890
Name: count, 

In [33]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

df['Revenue'] = df['Quantity'] * df['Price']


df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary',
         'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID',
    how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)

print("Export completed successfully!")
print("Final dataset shape:", df_final.shape)

Export completed successfully!
Final dataset shape: (779495, 18)


In [32]:
display(df_final.head())

print("Total rows:", len(df_final))
print("Total columns:", len(df_final.columns))

print("\nCustomer Segment Distribution:")
print(df_final['Segment'].value_counts())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,InvoiceMonth,Revenue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal,True
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2009-12-01,59.5,157,8,2433.28,3,4,4,Loyal,True
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,100.8,157,8,2433.28,3,4,4,Loyal,True
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2009-12-01,39.6,157,8,2433.28,3,4,4,Loyal,True


Total rows: 779495
Total columns: 18

Customer Segment Distribution:
Segment
Champions          483707
Loyal              155349
At Risk             76157
Lost                36956
New                 16363
Needs Attention     10963
Name: count, dtype: int64
